# 🐍 Day 3: Class Decorators

- Class decorators
- Descriptor protocol: __get__, __set__, __delete__
- Property-like descriptors
- Data vs non-data descriptors

## Class Decorators

Decorators can wrap classes too. They receive the class and return a modified class (or a new one).

In [ ]:
def add_repr(cls):
    def __repr__(self):
        attrs = ', '.join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(3, 4)
print(p)  # Point(x=3, y=4)

## Class Decorator: Add Methods

Use class decorators to inject methods, validate attributes, or register classes.

In [ ]:
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Database:
    def __init__(self):
        print("Connecting to DB...")

d1 = Database()
d2 = Database()
print(d1 is d2)

## Descriptor Protocol

Descriptors are objects that define `__get__`, `__set__`, or `__delete__`. They customize attribute access. `property` is a built-in descriptor.

In [ ]:
class Descriptor:
    def __get__(self, obj, objtype=None):
        print(f"__get__ called: obj={obj}, objtype={objtype}")
        return "value"

    def __set__(self, obj, value):
        print(f"__set__ called: obj={obj}, value={value}")

    def __delete__(self, obj):
        print(f"__delete__ called: obj={obj}")

class MyClass:
    x = Descriptor()

m = MyClass()
print(m.x)
m.x = 10
del m.x

## __get__ Signature

`__get__(self, obj, objtype=None)`: `obj` is the instance (None for class access), `objtype` is the class.

In [ ]:
class VerboseDescriptor:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, "<unset>")

    def __set__(self, obj, value):
        obj.__dict__[self.name] = value

class Person:
    name = VerboseDescriptor("name")

p = Person()
p.name = "Alice"
print(p.name)
print(Person.name)

## Data vs Non-Data Descriptors

- **Data descriptor**: defines `__set__` or `__delete__`. Takes precedence over instance dict.
- **Non-data descriptor**: only `__get__`. Instance attribute shadows it.

In [ ]:
class NonDataDescriptor:
    def __get__(self, obj, objtype=None):
        return "from descriptor"

class DataDescriptor:
    def __get__(self, obj, objtype=None):
        return obj.__dict__.get("_value", "default")

    def __set__(self, obj, value):
        obj.__dict__["_value"] = value

class C:
    nd = NonDataDescriptor()
    dd = DataDescriptor()

c = C()
print(c.nd)
c.nd = "instance value"
print(c.nd)  # instance shadows non-data descriptor

c.dd = "stored"
print(c.dd)  # descriptor's __set__ used

## Custom Property-Like Descriptor

Build a descriptor that validates or transforms values on get/set.

In [ ]:
class Positive:
    def __init__(self, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name, 0)

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(f"{self.name} must be >= 0")
        obj.__dict__[self.name] = value

class Rectangle:
    width = Positive("width")
    height = Positive("height")

    def __init__(self, width, height):
        self.width = width
        self.height = height

r = Rectangle(4, 5)
print(r.width, r.height)
# r.width = -1  # ValueError

## Typed Descriptor

Enforce type on attribute assignment.

In [ ]:
class Typed:
    def __init__(self, name, expected_type):
        self.name = name
        self.expected_type = expected_type

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} must be {self.expected_type.__name__}")
        obj.__dict__[self.name] = value

class Person:
    name = Typed("name", str)
    age = Typed("age", int)

p = Person()
p.name = "Alice"
p.age = 30
print(p.name, p.age)
# p.age = "thirty"  # TypeError

## Common Mistakes

- **Storing in descriptor**: Use `obj.__dict__[self.name]` not `self.value` (shared across instances)
- **Forgetting obj is None**: Class access returns descriptor when obj is None
- **Data vs non-data**: If you need to "override" from instance, use non-data
- **Class decorator returning wrong type**: Must return a class (or callable that acts like one)

## Exercises

In [ ]:
# Exercise 1: Create a class decorator that adds a __str__ method showing all attributes.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Create a descriptor that stores values in a dict with a prefix (e.g., _attr_name).
# YOUR CODE HERE

In [ ]:
# Exercise 3: Create a ReadOnly descriptor: __set__ raises AttributeError.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Create a descriptor that logs every get and set.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Create a class decorator that freezes a class (no new attributes after __init__).
# YOUR CODE HERE

In [ ]:
# Exercise 6: Implement a cached descriptor that computes value on first access and caches it.
# YOUR CODE HERE

## Key Takeaways

- Class decorators receive and return classes
- Descriptors: __get__, __set__, __delete__
- Data descriptors override instance dict; non-data don't
- Store per-instance data in obj.__dict__[name]

## 🔜 Next: Day 4 - Metaclasses

Tomorrow: type(), metaclass creation, __new__ vs __init__!